### Create Training data for binary classification: description of business sector - True False

In [1]:
import pandas as pd
import glob
import os
import ast
import json

import numpy as np
import pandas as pd
import os
import sys
import glob
import tqdm
import re
import random
from typing import List

sys.path.append("../../")
from sentence_splitter import split_text_into_sentences
#from BERT_classifier.Classify_report_with_BERT import classification_report_BERT
#from transformers import AutoModelForSequenceClassification, AutoTokenizer

#### Load Positives

In [2]:
positives_json_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/datasets/stoxx_600/JSONs"

In [3]:
df_overview = pd.read_csv("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/datasets/stoxx_600/stoxx_600_overview.csv", sep=";")
df_positives = df_overview[df_overview["description_page"].notna()]
df_positives

,Unnamed: 0,Name,Symbol,FactSet ID,Revenue - 2022 (in EUR),Revenue - 2023 (in EUR),Revenue - 2024 (in EUR),NACE,NACE_letter,Report,description_page
3,107,AAK AB,AAK-SE,AAK-SE,4741.086317,4010.433824,3939.34280627966,10.89,C,AAK AB1.pdf,3
6,94,ABB Ltd.,ABBN-CH,ABBN-CH,28073.948473,29665.651960,30586.3762461154,27.11,C,ABB Ltd.2.pdf,18
9,135,Accelleron Industries AG,ACLN-CH,ACLN-CH,742.680345,846.217532,NaN,28.11,C,Accelleron Industries AG1.pdf,7
10,296,Acciona SA,ANA-ES,ANA-ES,11195.000000,17021.000000,19190,41.20,F,Acciona SA2.pdf,7
11,365,Accor SA,AC-FR,AC-FR,4224.000000,5056.000000,5606,55.10,I,Accor SA1.pdf,5
...,...,...,...,...,...,...,...,...,...,...,...
394,69,Orkla ASA,ORK-NO,ORK-NO,5774.960695,5932.590483,6073.67720031738,10.89,C,Orkla ASA1.pdf,11
466,598,Scout24 SE,G24-DE,G24-DE,447.539000,509.114000,s,96.09,S,Scout24 SE3.pdf,41
472,286,Severn Trent Plc,SVT-GB,SVT-GB,2505.264229,2709.430915,NaN,36.00,E,Severn Trent Plc1.pdf,8
479,133,Siemens Energy AG,ENR-DE,ENR-DE,29005.000000,31119.000000,34465,27.33,C,Siemens Energy AG1.pdf,5


In [4]:
for i, row in df_positives.iterrows(): 
    path = row["Report"].replace(".pdf", ".json")
    with open(os.path.join(positives_json_path, path), "r") as f: 
        report_json = json.load(f)
    
    description_pages = ast.literal_eval(row["description_page"])
    description_pages = [description_pages] if isinstance(description_pages, int) else description_pages

    description_text = ""
    for description_page in description_pages:

        description_text += list(filter(lambda x: x["page"] == description_page + 1, report_json["pages"]))[0]["markdown"]

    print(row["Report"])
    print(description_text)
    print("-----" * 5 + "\n"*6)

    df_positives.loc[i, "Description"] = description_text

AAK AB1.pdf
## we do is about Making Better Happen ™ Everything

AAK specializes in plant-based oils and fats, the value-adding ingredients in many products people love to consume. We make these products better tasting, healthier, and more sustainable. In addition, we enhance their sensory experience - by giving the silkier mouthfeel in premium chocolate, the juicier texture in a plant-based burger, and a puffier appearance in a lower-fat pastry.

We can also optimize our customers' production and processes by substituting existing ingredients with plantbased equivalents that improve efficiency and enhance the performance and sustainability of the end product. AAK's value-adding solutions enable our customers to Making Better Happen ™ .

At the heart of AAK's offer is customer co-development, combining our desire to understand what Making Better Happen ™  means for each customer with the unique flexibility of our production assets and deep knowledge of products and industries, includin

/var/folders/fp/yhl61lbj3m73x3_17tsp1vrr0000gn/T/ipykernel_22847/2710329527.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_positives.loc[i, "Description"] = description_text


In [5]:
def get_tables(lines: list): 
    tables = []
    current_table = []

    for line in lines:
        if line.strip().startswith("|"):  # line belongs to a table
            current_table.append(line.strip())
        else:
            if current_table:  # table ended
                tables.append("\n".join(current_table))
                current_table = []

    # catch last table if file ends without empty lines
    if current_table:
        tables.append("\n".join(current_table))

    return tables

def preprocess_report(text: str) -> List[str]:

    # with open(pdf_path, "r") as f: 
    #     text = f.read()

    sentence_length = 1
    
    lines = text.split("\n")

    tables = get_tables(lines)

    # drop if condidtion is True
    conditions = [
        # filter images
        #lambda line: line == '<!-- image -->',
        
        #filter tables 
        #lambda line: (line[0] == "|" and line[-1] == "|") if len(line) > 1 else False, 

        # filter headers
        lambda line: line.strip()[0] == "#" if len(line) > 0 else True,

        # filter sentences
        #lambda line: "." not in line,
        
        # more than 50% is numbers
        #lambda line: sum(ch.isalpha() for ch in line) / len(line) < 0.5,

        # minimum 3 words 
        #lambda line: len(re.sub(r"[^a-zA-ZäöüÄÖÜß\s]", '', line).strip().split(" ")) < 3,

        # Minimum 2 Sentences
        #lambda line: sum([0 if len(sentence.split(" ")) < 3 else 1 for sentence in split_text_into_sentences(line, "en")]) < 2

    ]
    accepted_lines = [line for line in lines if not any(condition(line) for condition in conditions)]
    accepted_lines += tables

    chunks = []

    for line in accepted_lines: 
        sentences = split_text_into_sentences(line, language='en')
        sentences = [sentence.strip() for sentence in sentences]
        sentences = [sentence for sentence in sentences if sentence != ""]
        new_chunks = [(" ".join(sentences[i:i+sentence_length])).strip() for i in range(0, len(sentences), 3)]

        chunks += new_chunks
    
    if len(chunks) <= 1: 
        return chunks

    # if there is only one sentence in the last chunk, balance the two last chunks
    if len(split_text_into_sentences(chunks[-1], language = "en")) == 1: 
        last_two_chunks = chunks[-2] + " " + chunks[-1]
        chunks[-2] = last_two_chunks[0:(len(last_two_chunks) + 1) // 2]
        chunks[-1] = last_two_chunks[(len(last_two_chunks) + 1) // 2: (len(last_two_chunks)) - (len(last_two_chunks) + 1) // 2]

    chunks = [re.sub(r'\b\d+\.\d+\b', '', chunk) for chunk in chunks]
    chunks = [re.sub(r"[^a-zA-ZäöüÄÖÜß.\s]", '', chunk) for chunk in chunks]
    chunks = [re.sub(r"\s+", " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'\.{2,}', " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'^\d+\.\s*', " ", chunk) for chunk in chunks]
    chunks = [chunk.lower() for chunk in chunks]
    chunks = [chunk.strip() for chunk in chunks]

    return chunks

In [6]:
df_positives.loc[:,"lenght_desc"] = df_positives["Description"].apply(len)
df_positives

/var/folders/fp/yhl61lbj3m73x3_17tsp1vrr0000gn/T/ipykernel_22847/3719149439.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_positives.loc[:,"lenght_desc"] = df_positives["Description"].apply(len)


,Unnamed: 0,Name,Symbol,FactSet ID,Revenue - 2022 (in EUR),Revenue - 2023 (in EUR),Revenue - 2024 (in EUR),NACE,NACE_letter,Report,description_page,Description,lenght_desc
3,107,AAK AB,AAK-SE,AAK-SE,4741.086317,4010.433824,3939.34280627966,10.89,C,AAK AB1.pdf,3,## we do is about Making Better Happen ™ Every...,1424
6,94,ABB Ltd.,ABBN-CH,ABBN-CH,28073.948473,29665.651960,30586.3762461154,27.11,C,ABB Ltd.2.pdf,18,-\n\n## Who we are\n\nABB has a history of inn...,2907
9,135,Accelleron Industries AG,ACLN-CH,ACLN-CH,742.680345,846.217532,NaN,28.11,C,Accelleron Industries AG1.pdf,7,## Accelleron at a glance\n\nAccelleron's tech...,1491
10,296,Acciona SA,ANA-ES,ANA-ES,11195.000000,17021.000000,19190,41.20,F,Acciona SA2.pdf,7,## 2. BUSINESS MODEL: BUSINESS AS UNUSUAL\n\nA...,1569
11,365,Accor SA,AC-FR,AC-FR,4224.000000,5056.000000,5606,55.10,I,Accor SA1.pdf,5,"## Message from Sébastien Bazin, Chairman and ...",3503
...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,69,Orkla ASA,ORK-NO,ORK-NO,5774.960695,5932.590483,6073.67720031738,10.89,C,Orkla ASA1.pdf,11,"## Orkla's business areas in 2022\n\nIn 2022, ...",753
466,598,Scout24 SE,G24-DE,G24-DE,447.539000,509.114000,s,96.09,S,Scout24 SE3.pdf,41,≡\n\n## Grundlagen des Konzerns\n\n## Geschäft...,3264
472,286,Severn Trent Plc,SVT-GB,SVT-GB,2505.264229,2709.430915,NaN,36.00,E,Severn Trent Plc1.pdf,8,## OUR BUSINESS MODEL UNDERSTANDING OUR WORLD\...,2100
479,133,Siemens Energy AG,ENR-DE,ENR-DE,29005.000000,31119.000000,34465,27.33,C,Siemens Energy AG1.pdf,5,## Our business areas\n\n## Digital Industries...,4527


In [7]:
paragraphs = []

for i, row in df_positives.iterrows(): 
    paragraphs.extend(preprocess_report(row["Description"]))

In [8]:
paragraphs = [p for p in paragraphs if p != ""]
paragraphs = [p for p in paragraphs if len(p) > 30]
len(paragraphs)

759

In [9]:
df_paragraphs_positive = pd.DataFrame(paragraphs)
df_paragraphs_positive.to_csv("/Users/hendrikweichel/Desktop/positive_data.csv")

In [17]:
positive_paragraphs_selected = pd.read_csv("/Users/hendrikweichel/Desktop/positive_data_selected.csv", sep=";", index_col=0)
positive_paragraphs_selected = positive_paragraphs_selected.dropna()

#### Load Negatives

In [16]:
negative_paragraphs = []

for i, row in df_positives.iterrows(): 
    path = row["Report"].replace(".pdf", ".json")
    with open(os.path.join(positives_json_path, path), "r") as f: 
        report_json = json.load(f)
    description_pages = ast.literal_eval(row["description_page"])
    description_pages = [description_pages] if isinstance(description_pages, int) else description_pages

    description_text = ""
    for description_page in description_pages:

        description_text += list(filter(lambda x: x["page"] != description_page + 1, report_json["pages"]))[np.random.randint(len(report_json["pages"])-1)]["markdown"]

    print(row["Report"])
    print(description_text)
    print("-----" * 5 + "\n"*6)

    #df_positives.loc[i, "Description"] = description_text

    negative_paragraphs.extend(preprocess_report(description_text))

AAK AB1.pdf
## Consolidated Changes in Shareholders' Equity

|                                         | Attributable to the Parent's shareholders   | Attributable to the Parent's shareholders   | Attributable to the Parent's shareholders   | Non-controlling   |              |
|-----------------------------------------|---------------------------------------------|---------------------------------------------|---------------------------------------------|-------------------|--------------|
| SEK million                             | Share capital                               | Reserves                                    | Retained profit                             | interests         | Total equity |
| Opening balance as at January 1, 2021   | 426                                         | -1,456                                      | 10,729                                      | 39                | 9,738        |
| Profit for the year                     | -                          

In [12]:
negative_paragraphs = [p for p in negative_paragraphs if p != ""]
negative_paragraphs = [p for p in negative_paragraphs if len(p) > 60]
len(negative_paragraphs)

698

In [13]:
negative_paragraphs

['composition of business segments power grids discontinued robotics and discrete automation motion electrification process automation in crores total',
 'revenues results net profit assets liabilities in crores depreciation amortisation',
 'composition of business segments power grids discontinued robotics and discrete automation motion electrification process automation in crores total',
 'revenues results net profit assets liabilities in crores depreciation amortisation',
 'the amounts recognized in accumulated other comprehensive income aoci at december before taxes were',
 'the following amounts were recognized in the companys consolidated and combined balance sheet as at december and classified as noncurrent assets',
 'the following assumptions were used to determine the projected benefit obligation at december weighted average',
 'for the companys benefit plans the discount rate used at each measurement date is set based on a highquality corporate bond yield curve reflecting the

In [14]:
df_negative_paragraphs = pd.DataFrame(negative_paragraphs).drop_duplicates()
df_negative_paragraphs.to_csv("/Users/hendrikweichel/Desktop/negative_data.csv")

In [15]:
len(df_negative_paragraphs)

647

In [18]:
df_negative_paragraphs["label"] = False
positive_paragraphs_selected["label"] = True

In [21]:
df_negative_paragraphs

,0,label
0,composition of business segments power grids d...,False
1,revenues results net profit assets liabilities...,False
4,the amounts recognized in accumulated other co...,False
5,the following amounts were recognized in the c...,False
6,the following assumptions were used to determi...,False
...,...,...
693,additional fees invoiced for statutory audit i...,False
694,siemens as presents its income statement based...,False
695,following is the comparative summary of our fi...,False
696,during the year under review the company sales...,False


In [26]:
df_negative_paragraphs.columns, positive_paragraphs_selected.columns

(Index([0, 'label'], dtype='object'), Index(['0', 'label'], dtype='object'))

In [29]:
df_negative_paragraphs = df_negative_paragraphs.rename(columns={0:"text"})
positive_paragraphs_selected = positive_paragraphs_selected.rename(columns={"0":"text"})

In [40]:
df_full = pd.concat((df_negative_paragraphs, positive_paragraphs_selected), axis=0)

In [41]:
df_full = df_full.reset_index(drop=True)
df_full

,text,label
0,composition of business segments power grids d...,False
1,revenues results net profit assets liabilities...,False
2,the amounts recognized in accumulated other co...,False
3,the following amounts were recognized in the c...,False
4,the following assumptions were used to determi...,False
...,...,...
1153,we are building a customerfocused organization...,True
1154,we are developing tiered offerings with multip...,True
1155,we are driving new sustainable growth areas to...,True
1156,we are creating a digital front and backend em...,True


In [43]:
df_full.to_csv("/Users/hendrikweichel/Desktop/full_data_comp_description.csv")

In [44]:
from sklearn.model_selection import train_test_split

# Split full_df into train (60%) and temp (40%)
train_df, temp_df = train_test_split(df_full, test_size=0.4, random_state=42)

# Split temp into test (20%) and validation (20%)
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Print the sizes of each split
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}, Validation size: {len(val_df)}")

Train size: 694, Test size: 232, Validation size: 232


In [45]:
train_df.to_csv("/Users/hendrikweichel/Desktop" + "/train_data.csv", index=False)
val_df.to_csv("/Users/hendrikweichel/Desktop" + "/val_data.csv", index=False)
test_df.to_csv("/Users/hendrikweichel/Desktop" + "/test_data.csv", index=False)